<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 01
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Phân tích thị trường mỹ phẩm nội vs ngoại trên Tiki"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  THU THẬP BỘ DỮ LIỆU TỪ TIKI
</div>

## **1. Khám phá cấu trúc danh mục**
### **1.1. Mục tiêu**
Trước khi tiến hành thu thập dữ liệu sản phẩm trên quy mô lớn, nhóm cần xác định chính xác cấu trúc phân cấp ngành hàng của Tiki. Mục tiêu của bước này là trích xuất các mã định danh (ID) và đường dẫn định danh (URL Key) của các danh mục con thuộc ngành hàng "Làm đẹp - Sức khỏe". Việc xác định đúng ID danh mục là yếu tố tiên quyết để bộ lọc dữ liệu đi đúng hướng, tránh thu thập các sản phẩm không liên quan.

### **1.2. Thư viện sử dụng**
* `requests`: Thư viện chính để gửi các yêu cầu HTTP đến API của Tiki.
* `json`: Xử lý dữ liệu trả về dưới định dạng JSON.
* `tqdm`: Hiển thị thanh tiến độ trực quan (dành cho các bước cào dữ liệu lặp lại sau này).

### **1.3. Quy trình thực thi**
1. **Khởi tạo môi trường:** Cài đặt các thư viện cần thiết trực tiếp vào kernel của Notebook.
2. **Cấu hình Headers:** Thiết lập `User-Agent` giả lập trình duyệt và `Referer` từ Tiki để tránh bị hệ thống chặn truy cập (Anti-crawling).
3. **Truy vấn API Danh mục:**
   - Gửi yêu cầu GET đến endpoint: `https://tiki.vn/api/v2/categories`.
   - Sử dụng tham số `parent_id: 1520` (Mã ID gốc của ngành hàng Làm đẹp - Sức khỏe trên Tiki).
4. **Phân tích kết quả:**
   - Duyệt qua cây danh mục để lấy thông tin các danh mục cấp 1 và cấp 2.
   - In ra danh sách ID và tên danh mục để làm căn cứ thiết lập bộ lọc thu thập dữ liệu ở bước tiếp theo.


In [1]:
import sys
!{sys.executable} -m pip install tqdm pandas requests numpy
import requests, json

HEADERS = {"User-Agent": "Mozilla/5.0", "Referer": "https://tiki.vn/"}

res = requests.get(
    "https://tiki.vn/api/v2/categories",
    headers=HEADERS,
    params={"include": "children", "parent_id": 1520}
)

data = res.json()
print("Status:", res.status_code)
print("Keys:", list(data.keys()))

categories = data.get("data", [])
print(f"\nSố danh mục con: {len(categories)}\n")

for cat in categories:
    print(f"[{cat['id']}] {cat['name']}  →  urlKey: {cat.get('url_key','')}")
    # In sub-category cấp 2 nếu có
    for sub in cat.get("children", []):
        print(f"    [{sub['id']}] {sub['name']}  →  urlKey: {sub.get('url_key','')}")



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Status: 200
Keys: ['data', 'show_max']

Số danh mục con: 12

[1582] Chăm sóc da mặt  →  urlKey: cham-soc-da-mat
    [1601] Mặt nạ  →  urlKey: mat-na-cac-loai
    [5872] Xịt khoáng   →  urlKey: xit-khoang
    [3424] Chăm sóc vùng da mắt  →  urlKey: cham-soc-vung-da-mat
    [2347] Nước hoa hồng, toner  →  urlKey: nuoc-hoa-hong-toner
    [3422] Kem chống nắng  →  urlKey: kem-chong-nang
    [3426] Sản phẩm trị mụn và sẹo  →  urlKey: san-pham-tri-mun-va-seo
    [17170] Kem và sữa dưỡng da  →  urlKey: kem-va-sua-duong-da
    [17174] Sản phẩm chăm sóc da mặt khác  →  urlKey: san-pham-cham-soc-da-mat-khac
    [53350] Serum  →  urlKey: serum
    [8227] Bộ chăm sóc da mặt  →  urlKey: bo-cham-soc-da-mat
    [1583] Sữa rửa mặt  →  urlKey: sua-rua-mat
    [8206] Tẩy tế bào chết  →  urlKey: tay-te-bao-chet
    [11605] Dung dịch tẩy trang  →  urlKey: dung-dich-tay-trang
    [5893] Chống lão hóa da  →  urlKey: chong-lao-hoa-da
    [5865] Bông tẩy trang  →  urlKey: bong-tay-trang
    [70250] Giấy thấm 

## **2. THIẾT LẬP THAM SỐ VÀ DANH MỤC MỤC TIÊU**
### **2.1. Mục tiêu**
Sau khi đã khám phá cấu trúc danh mục ở bước 1, nhóm tiến hành thiết lập các tham số kỹ thuật và xác định phạm vi thu thập dữ liệu cụ thể. Thay vì cào toàn bộ ngành hàng Làm đẹp, nhóm thực hiện phương pháp lọc thủ công để chọn ra các danh mục mỹ phẩm thực sự, loại bỏ các mục không liên quan như thiết bị y tế, bỉm tã hay máy massage để đảm bảo độ tập trung của tập dữ liệu.

### **2.2. Cấu hình kỹ thuật**
Để đảm bảo quá trình thu thập dữ liệu diễn ra ổn định và không bị hệ thống Tiki chặn (Block IP), nhóm thiết lập bộ Headers mô phỏng trình duyệt người dùng thật:
* **User-Agent:** Cung cấp thông tin về hệ điều hành và trình duyệt.
* **Referer:** Khai báo nguồn gốc yêu cầu từ trang chủ Tiki.
* **Accept-Language:** Ưu tiên ngôn ngữ tiếng Việt để nhận được dữ liệu chuẩn xác nhất.

### **2.3. Danh sách danh mục mục tiêu**
Nhóm đã phân loại và lựa chọn **31 danh mục con** trọng tâm, chia thành 5 nhóm sản phẩm lớn để phục vụ bài toán so sánh:
1. **Chăm sóc da mặt (Face Care):** Sữa rửa mặt, Serum, Kem dưỡng, Kem chống nắng, Toner, Mặt nạ, Trị mụn, Chống lão hóa, Tẩy tế bào chết, Xịt khoáng.
2. **Trang điểm (Makeup):** Son môi, Trang điểm mặt, Trang điểm mắt, Dụng cụ trang điểm, Bộ trang điểm, Trang điểm khác.
3. **Chăm sóc cơ thể (Body Care):** Sữa tắm, Dưỡng thể, Kem chống nắng (body), Tẩy tế bào chết body, Dung dịch vệ sinh, Tẩy lông, Lăn khử mùi.
4. **Nước hoa (Fragrance):** Nước hoa nữ và nước hoa nam.
5. **Chăm sóc tóc (Hair Care):** Dầu gội & xả, Dưỡng tóc, Tạo kiểu tóc, Thuốc nhuộm, Bộ chăm sóc tóc.

Mỗi danh mục được định nghĩa bởi bộ ba thông tin: `{id, urlKey, label}`. Trong đó, `label` sẽ được sử dụng làm nhãn phân loại chính trong tập dữ liệu cuối cùng.

> **Lưu ý:** `urlKey` phải khớp chính xác với ID danh mục tương ứng để API Tiki trả về đúng tập sản phẩm. Hai tham số này hoạt động song song như một cơ chế xác thực kép.


In [2]:
import requests
import pandas as pd
import time
from tqdm import tqdm

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer":         "https://tiki.vn/",
    "Accept":          "application/json, text/plain, */*",
    "Accept-Language": "vi,en-US;q=0.9,en;q=0.8",
}

# Chỉ lấy category mỹ phẩm thực sự
# Lưu ý: mỗi entry cần đảm bảo id và urlKey là cặp chính xác
CATEGORIES = [
    # ── Chăm sóc da mặt (Face Care) ──────────────────────────
    {"id": 1583,  "urlKey": "sua-rua-mat",                  "label": "Sữa rửa mặt"},
    {"id": 11605, "urlKey": "dung-dich-tay-trang",          "label": "Tẩy trang"},
    {"id": 2347,  "urlKey": "nuoc-hoa-hong-toner",          "label": "Toner"},
    {"id": 53350, "urlKey": "serum",                        "label": "Serum"},
    {"id": 17170, "urlKey": "kem-va-sua-duong-da",          "label": "Kem dưỡng da"},
    {"id": 1601,  "urlKey": "mat-na-cac-loai",              "label": "Mặt nạ"},
    {"id": 3422,  "urlKey": "kem-chong-nang",               "label": "Kem chống nắng (mặt)"},
    {"id": 3426,  "urlKey": "san-pham-tri-mun-va-seo",      "label": "Trị mụn & sẹo"},
    {"id": 5893,  "urlKey": "chong-lao-hoa-da",             "label": "Chống lão hóa"},
    {"id": 8206,  "urlKey": "tay-te-bao-chet",              "label": "Tẩy tế bào chết mặt"},
    {"id": 5872,  "urlKey": "xit-khoang",                   "label": "Xịt khoáng"},

    # ── Trang điểm (Makeup) ───────────────────────────────────
    {"id": 1587,  "urlKey": "son-trang-diem-moi",           "label": "Son môi"},
    {"id": 1585,  "urlKey": "trang-diem-mat",               "label": "Trang điểm mặt"},
    {"id": 1586,  "urlKey": "trang-diem-mat-2",             "label": "Trang điểm mắt"},
    {"id": 1589,  "urlKey": "dung-cu-trang-diem",           "label": "Dụng cụ trang điểm"},
    {"id": 8228,  "urlKey": "bo-trang-diem",                "label": "Bộ trang điểm"},
    {"id": 53354, "urlKey": "trang-diem-khac",              "label": "Trang điểm khác"},

    # ── Chăm sóc cơ thể (Body Care) ──────────────────────────
    {"id": 8212,  "urlKey": "tam-va-dung-cu-tam",           "label": "Sữa tắm"},
    {"id": 1610,  "urlKey": "duong-the",                    "label": "Dưỡng thể"},
    {"id": 1615,  "urlKey": "san-pham-chong-nang",          "label": "Kem chống nắng (thể)"},
    {"id": 8220,  "urlKey": "tay-te-bao-chet-co-the",       "label": "Tẩy tế bào chết body"},
    {"id": 1752,  "urlKey": "dung-dich-ve-sinh",            "label": "Dung dịch vệ sinh"},
    {"id": 3449,  "urlKey": "san-pham-tay-long",            "label": "Sản phẩm tẩy lông"},
    {"id": 17162, "urlKey": "lan-xit-khu-mui",              "label": "Lăn, xịt khử mùi"},

    # ── Nước hoa (Fragrance) ──────────────────────────────────
    {"id": 1636,  "urlKey": "nuoc-hoa-nu",                  "label": "Nước hoa nữ"},
    {"id": 1637,  "urlKey": "nuoc-hoa-nam",                 "label": "Nước hoa nam"},

    # ── Chăm sóc tóc (Hair Care) ──────────────────────────────
    {"id": 8222,  "urlKey": "dau-goi-dau-xa",               "label": "Dầu gội & xả"},
    {"id": 7065,  "urlKey": "duong-toc-u-toc",              "label": "Dưỡng tóc"},
    {"id": 1620,  "urlKey": "tao-kieu-toc",                 "label": "Tạo kiểu tóc"},
    {"id": 1623,  "urlKey": "thuoc-nhuom",                  "label": "Thuốc nhuộm tóc"},
    {"id": 7061,  "urlKey": "bo-cham-soc-toc",              "label": "Bộ chăm sóc tóc"},
]

## **3. XÂY DỰNG CÁC HÀM THU THẬP VÀ TRÍCH XUẤT THÔNG TIN**
Sau khi đã có danh sách các Category mục tiêu, nhóm tiến hành xây dựng hai hàm quan trọng để tự động hóa việc lấy dữ liệu và chuẩn hóa thông tin ngay từ bước thô.

### **3.1. Hàm `fetch_page`: Tương tác với API Tiki**
Hàm này thực hiện gửi yêu cầu lấy dữ liệu sản phẩm từ API của Tiki.

* **Logic xử lý tham số:**
    * `sort = top_seller`: Nhóm chọn tiêu chí sắp xếp này để ưu tiên lấy những sản phẩm có dữ liệu bán hàng thực tế. Đây là nguồn dữ liệu quan trọng nhất để phân tích sự ưu tiên của người tiêu dùng.
    * `version = home-personalized`: Sử dụng phiên bản API này để lấy được nhiều thông tin ẩn hơn, đặc biệt là cờ nhập khẩu `is_imported` — trường dữ liệu cốt lõi cho bài toán phân tích Nội/Ngoại. *(Lưu ý: đúng chính tả là `home-personalized`, không phải `home-persionalized`.)*
    * `timeout = 10`: Thiết lập thời gian chờ để tránh chương trình bị treo nếu server Tiki phản hồi chậm.

### **3.2. Hàm `parse_product`: Trích xuất và ánh xạ dữ liệu**
Dữ liệu thô từ Tiki trả về rất nhiều thông tin dư thừa. Hàm này giúp lọc lấy đúng những "nguyên liệu" cần thiết cho bài toán phân tích.

* **Trọng tâm dữ liệu:**
    * **Nhóm định danh:** ID, Tên, Thương hiệu, Nhà bán.
    * **Nhóm chỉ số kinh doanh:** Giá bán, Giá gốc (để tính % giảm giá), Lượt bán (`sold_count`), Điểm đánh giá (`rating`) và lượt nhận xét.
    * **Nhóm thông tin quan trọng** (Dùng cho bài toán Nội/Ngoại):
        * `origin`: Xuất xứ — trường quan trọng nhất để phân loại hàng nội/ngoại.
        * `is_imported`: Cờ đánh dấu hàng nhập khẩu, lấy từ object `amplitude` bên trong `visible_impression_info`.
        * `is_from_official_store`: Xác định gian hàng chính hãng (Tiki Mall).
        * `badges_new`: Kiểm tra nhãn `authentic_brand` để tăng độ tin cậy khi xác thực chính hãng.

* **Kỹ thuật xử lý đặc biệt:**
    * **`quantity_sold` không đồng nhất:** Tiki đôi khi trả về trường này là một số nguyên, đôi khi là một object `{"value": N, ...}`. Nhóm viết thêm logic kiểm tra `isinstance` để đảm bảo luôn lấy được con số chính xác trong cả hai trường hợp.
    * **Giá trị mặc định an toàn:** Sử dụng `.get()` với giá trị mặc định (0 hoặc False) cho mọi trường để tránh `KeyError` khi sản phẩm thiếu thông tin.


In [3]:
# ─────────────────────────────────────────────────────────────
# HÀM 1: fetch_page – Gọi API Tiki lấy danh sách sản phẩm
# ─────────────────────────────────────────────────────────────
def fetch_page(cat_id, url_key, page, limit=50):
    """
    Lấy danh sách sản phẩm từ 1 trang của 1 danh mục trên Tiki.
    Trả về list các dict sản phẩm thô, hoặc [] nếu lỗi / hết trang.
    """
    try:
        res = requests.get(
            "https://tiki.vn/api/v2/products",
            headers=HEADERS,
            params={
                "category_id":  cat_id,
                "urlKey":       url_key,
                "page":         page,
                "limit":        limit,
                "sort":         "top_seller",
                "version":      "home-personalized",
                "aggregations": 2,
            },
            timeout=10,
        )
        if res.status_code != 200:
            return []
        return res.json().get("data", [])
    except Exception as e:
        print(f"  Lỗi trang {page}: {e}")
        return []


# ─────────────────────────────────────────────────────────────
# HÀM 2: parse_product – Trích xuất các trường cần thiết
# ─────────────────────────────────────────────────────────────
def parse_product(p, label):
    """
    Ánh xạ dữ liệu thô từ API Tiki sang các trường phân tích.
    `label` là tên danh mục do nhóm tự gán (vd: 'Sữa rửa mặt').
    """
    # Lấy object amplitude chứa is_imported (nằm lồng sâu trong response)
    amp         = p.get("visible_impression_info", {}).get("amplitude", {})
    badge_codes = [b.get("code", "") for b in p.get("badges_new", [])]
    price       = p.get("price") or 0
    orig        = p.get("original_price") or price

    # Xử lý sold_count: Tiki trả về dạng int hoặc dict {"value": N, ...}
    raw_sold = p.get("quantity_sold")
    if isinstance(raw_sold, dict):
        sold_count = raw_sold.get("value", 0) or 0
    else:
        sold_count = raw_sold or 0

    return {
        # ── Nhóm định danh ──────────────────────────────────────
        "product_id":          p.get("id"),                            # ID định danh duy nhất trên hệ thống Tiki
        "name":                p.get("name"),                          # Tên đầy đủ hiển thị cho người mua
        "brand_id":            p.get("brand_id"),                      # ID thương hiệu
        "brand_name":          p.get("brand_name"),                    # Tên thương hiệu (vd: Cocoon, Anessa, L'Oréal)
        "seller_id":           p.get("seller_id"),                     # ID nhà bán hàng
        "seller_name":         p.get("seller_name"),                   # Tên gian hàng (vd: Tiki Trading, Guardian)
        "category":            label,                                  # Nhãn danh mục do nhóm tự quy định
        "primary_category":    p.get("primary_category_name"),         # Danh mục chính trong phân cấp Tiki

        # ── Nhóm chỉ số kinh doanh ──────────────────────────────
        "price":               price,                                  # Giá bán thực tế (đã áp dụng khuyến mãi)
        "original_price":      orig,                                   # Giá niêm yết ban đầu chưa giảm
        "discount_rate":       p.get("discount_rate", 0),              # Tỉ lệ giảm giá hiện tại (%)
        "rating":              p.get("rating_average", 0),             # Điểm đánh giá trung bình (thang 5)
        "review_count":        p.get("review_count", 0),               # Số lượng nhận xét của sản phẩm
        "sold_count":          sold_count,                             # Số lượng đã bán thành công

        # ── Nhóm thông tin cốt lõi cho bài toán Nội vs Ngoại ───
        "origin":              p.get("origin"),                        # Quốc gia xuất xứ (trường quan trọng nhất)
        "is_imported":         amp.get("is_imported"),                 # Cờ hàng nhập khẩu từ amplitude object
        "is_authentic":        p.get("is_authentic", 0),               # Xác nhận chính hãng (0 hoặc 1)
        "is_official_store":   p.get("is_from_official_store", False), # Gian hàng chính hãng (Tiki Mall)
        "tiki_verified":       p.get("tiki_verified", 0),              # Tiki đã kiểm tra và xác thực thông tin
        "has_authentic_badge": "authentic_brand" in badge_codes,       # Nhãn cam kết thương hiệu chính hãng
        "availability":        p.get("availability", 0),               # Trạng thái (1: Còn hàng, 3: Hết, 4: Ngừng KD)
    }


## **4. TIẾN HÀNH THU THẬP DỮ LIỆU**
### **4.1. Mục tiêu**
Sau khi đã chuẩn bị đầy đủ, nhóm tiến hành giai đoạn thu thập dữ liệu thực tế trên toàn bộ các danh mục đã chọn. Mục tiêu là xây dựng một tập dữ liệu thô đủ lớn (quy mô kỳ vọng > 5.000 sản phẩm) để đảm bảo tính đại diện và độ tin cậy cho các phân tích thống kê về sau.

### **4.2. Chiến lược thực hiện**
Nhóm sử dụng cấu trúc vòng lặp lồng nhau để đảm bảo quét triệt để các ngóc ngách của dữ liệu:
1. **Duyệt Danh mục (Outer Loop):** Lần lượt đi qua từng mục trong danh sách `CATEGORIES` đã lọc thủ công.
2. **Quét Trang (Inner Loop):** Đối với mỗi mục, tiến hành lật từng trang kết quả để lấy danh sách sản phẩm.
3. **Trích xuất & Lưu trữ:** Dữ liệu từ mỗi trang được hàm `parse_product` xử lý ngay lập tức và đưa vào danh sách tổng `all_products`.

### **4.3. Các cơ chế kiểm soát và bảo vệ hệ thống**
Để bộ cào hoạt động ổn định trên tập dữ liệu lớn mà không gặp sự cố, nhóm đã áp dụng 3 quy tắc kỹ thuật sau:

* **Giới hạn `MAX_PAGES = 200`:** Ngưỡng an toàn để tránh việc chương trình chạy vô hạn hoặc vượt quá giới hạn phân trang của phía Server Tiki.
* **Cơ chế Early Stopping:** Nếu một trang bất kỳ trả về kết quả rỗng (không còn sản phẩm), chương trình tự động ngắt vòng lặp của danh mục đó để chuyển ngay sang mục tiếp theo, giúp tối ưu hóa thời gian thực thi.
* **Chính sách Polite Crawling:** Sử dụng `time.sleep(1.0)` để tạo khoảng nghỉ 1 giây giữa mỗi lần gửi yêu cầu trang. Điều này giúp giảm thiểu nguy cơ bị hệ thống Anti-bot nhận diện, đồng thời không gây áp lực quá tải lên máy chủ Tiki.

### **4.4. Giám sát tiến độ**
Nhóm thiết lập hệ thống in log thời gian thực giúp theo dõi sát sao quá trình:
* Hiển thị số lượng sản phẩm mới thu được sau mỗi trang.
* Thống kê tổng số sản phẩm của từng danh mục.
* Cập nhật quy mô tổng thể của toàn bộ tập dữ liệu.


In [4]:
# Crawl toàn bộ danh mục
all_products = []
MAX_PAGES    = 200

for cat in CATEGORIES:
    print(f"\n>>> [{cat['id']}] {cat['label']}")
    cat_count = 0

    for page in range(1, MAX_PAGES + 1):
        items = fetch_page(cat["id"], cat["urlKey"], page)

        if not items:
            print(f"  Trang {page}: hết dữ liệu → dừng danh mục này")
            break

        for p in items:
            all_products.append(parse_product(p, cat["label"]))

        cat_count += len(items)
        print(f"  Trang {page:>3}: +{len(items):>2} | Danh mục: {cat_count:>4} | Tổng: {len(all_products):>6}")
        time.sleep(1.0)   # Polite crawling: nghỉ 1 giây giữa các trang

print(f"\n{'='*50}")
print(f"Thu thập xong. Tổng bản ghi thô: {len(all_products):,}")



>>> [1583] Sữa rửa mặt
  Trang   1: +40 | Danh mục:   40 | Tổng:     40
  Trang   2: +40 | Danh mục:   80 | Tổng:     80
  Trang   3: +40 | Danh mục:  120 | Tổng:    120
  Trang   4: +40 | Danh mục:  160 | Tổng:    160
  Trang   5: +40 | Danh mục:  200 | Tổng:    200
  Trang   6: +40 | Danh mục:  240 | Tổng:    240
  Trang   7: +40 | Danh mục:  280 | Tổng:    280
  Trang   8: +40 | Danh mục:  320 | Tổng:    320
  Trang   9: +40 | Danh mục:  360 | Tổng:    360
  Trang  10: +40 | Danh mục:  400 | Tổng:    400
  Trang  11: +39 | Danh mục:  439 | Tổng:    439
  Trang  12: +40 | Danh mục:  479 | Tổng:    479
  Trang  13: +40 | Danh mục:  519 | Tổng:    519
  Trang  14: +40 | Danh mục:  559 | Tổng:    559
  Trang  15: +40 | Danh mục:  599 | Tổng:    599
  Trang  16: +26 | Danh mục:  625 | Tổng:    625
  Trang 17: hết dữ liệu → dừng danh mục này

>>> [11605] Tẩy trang
  Trang   1: +40 | Danh mục:   40 | Tổng:    665
  Trang   2: +40 | Danh mục:   80 | Tổng:    705
  Trang   3: +40 | Danh mục

## **5. TỔNG KẾT VÀ LƯU TRỮ DỮ LIỆU THÔ**
### **5.1. Mục tiêu**
Sau khi hoàn tất quá trình cào dữ liệu lặp lại qua hàng trăm danh mục, bước cuối cùng là tập hợp tất cả về một cấu trúc bảng duy nhất, xử lý các bản ghi trùng lặp và lưu trữ bền vững để sẵn sàng cho giai đoạn Tiền xử lý.

### **5.2. Xử lý trùng lặp**
Trong cấu trúc của Tiki, một sản phẩm có thể xuất hiện ở nhiều danh mục khác nhau. Nhóm sử dụng phương thức `drop_duplicates(subset="product_id")` để đảm bảo mỗi sản phẩm chỉ xuất hiện duy nhất một lần dựa trên mã định danh, tránh làm sai lệch các kết quả thống kê như tổng doanh thu hay lượt bán về sau.

### **5.3. Lưu trữ và định dạng**
* **Định dạng:** Dữ liệu được lưu dưới dạng file `.csv` trong thư mục `data`.
* **Mã hóa (Encoding):** Sử dụng `utf-8-sig`. Đây là kỹ thuật quan trọng để đảm bảo khi mở file CSV bằng Excel, các ký tự tiếng Việt (tên sản phẩm, thương hiệu) vẫn hiển thị chính xác mà không bị lỗi font.

### **5.4. Kiểm định nhanh chất lượng dữ liệu**
Ngay sau khi lưu file, nhóm thực hiện in ra bản báo cáo nhanh để đánh giá sơ bộ chất lượng dữ liệu thô trên 4 chiều:
1. **Quy mô dữ liệu:** Tổng số sản phẩm thực tế thu được sau khi loại bỏ trùng lặp.
2. **Phân bố danh mục:** Kiểm tra xem số lượng sản phẩm giữa các nhóm có bị lệch quá mức không.
3. **Thống kê Origin:** Xem xét Top 15 quốc gia và tỉ lệ thiếu thông tin `origin` — định hình khối lượng công việc cần làm ở bước làm sạch dữ liệu.
4. **Kiểm tra `is_imported`:** Xác nhận trường này được điền đầy đủ, vì đây là trường được dùng để suy luận xuất xứ khi `origin` bị thiếu trong quá trình tiền xử lý.


In [5]:
# ─────────────────────────────────────────────────────────────
# LƯU FILE VÀ KIỂM ĐỊNH NHANH
# ─────────────────────────────────────────────────────────────
df = pd.DataFrame(all_products).drop_duplicates(subset="product_id")

path = "../data"
df.to_csv(f"{path}/tiki_cosmetics_raw.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"  ĐÃ LƯU: tiki_cosmetics_raw.csv")
print(f"{'='*50}")
print(f"Tổng sản phẩm (sau dedup) : {len(df):,}")
print(f"Số cột                    : {len(df.columns)}")
print(f"Tên các cột               : {list(df.columns)}")

print(f"\n--- Phân bố theo danh mục ---")
print(df["category"].value_counts().to_string())

print(f"\n--- Top 15 xuất xứ ---")
print(df["origin"].value_counts().head(15).to_string())

print(f"\n--- Tỉ lệ thiếu dữ liệu ---")
missing = (df.isnull().mean() * 100).round(1)
missing_cols = missing[missing > 0].sort_values(ascending=False)
if len(missing_cols) > 0:
    print(missing_cols.to_string())
else:
    print("Không có cột nào thiếu dữ liệu.")

print(f"\n--- Kiểm tra is_imported (cờ quan trọng cho bài toán Nội/Ngoại) ---")
print(df["is_imported"].value_counts(dropna=False).to_string())
print(f"Tỉ lệ thiếu is_imported: {df['is_imported'].isna().mean():.1%}")



  ĐÃ LƯU: tiki_cosmetics_raw.csv
Tổng sản phẩm (sau dedup) : 8,041
Số cột                    : 21
Tên các cột               : ['product_id', 'name', 'brand_id', 'brand_name', 'seller_id', 'seller_name', 'category', 'primary_category', 'price', 'original_price', 'discount_rate', 'rating', 'review_count', 'sold_count', 'origin', 'is_imported', 'is_authentic', 'is_official_store', 'tiki_verified', 'has_authentic_badge', 'availability']

--- Phân bố theo danh mục ---
category
Kem dưỡng da            888
Serum                   677
Sữa rửa mặt             625
Dầu gội & xả            563
Kem chống nắng (mặt)    499
Dụng cụ trang điểm      474
Sữa tắm                 469
Nước hoa nữ             435
Mặt nạ                  412
Tẩy trang               283
Trang điểm mắt          279
Trang điểm mặt          261
Nước hoa nam            252
Dưỡng thể               241
Toner                   237
Son môi                 212
Dưỡng tóc               205
Lăn, xịt khử mùi        181
Dung dịch vệ sinh 